# Кластеризация текстов

**Задача.** Дан набор объектов без меток — разреженная матрица размера **14 590 × 77 888**. Нужно разбить объекты на группы (кластеры) и для каждого проставить номер группы.

Это **обучение без учителя**: правильных ответов в данных нет. Модель сама ищет «кучки» похожих объектов. Опора — **гипотеза компактности**: близкие объекты обычно принадлежат одному классу.

**Данные** (не входят в репозиторий, выдаются в курсе):
- `train.npz` — разреженная матрица признаков;
- `sample_submission.csv` — пример файла ответа.

**Формат ответа:** `ID,TARGET`.

## Метрика — Adjusted Rand Score (ARS)

У организаторов есть настоящие классы, ARS сравнивает с ними наше разбиение. Важно не **как** мы назвали группы, а **кто с кем** оказался вместе:

| Настоящие классы | Наш ответ | ARS |
|---|---|---|
| `[A, A, B, B]` | `[0, 0, 1, 1]` | 1.0 — идеально |
| `[A, A, B, B]` | `[1, 1, 0, 0]` | 1.0 — номера другие, пары те же |
| `[A, A, B, B]` | `[0, 1, 0, 1]` | −0.5 — пары перепутаны |

ARS = 1 — полное совпадение, ≈ 0 — как случайное разбиение. Посчитать ARS у себя нельзя: для `adjusted_rand_score(labels_true, labels_pred)` нужны правильные метки, а они есть только у организаторов. Поэтому оценку даёт лидерборд.

## Итог

| # | Решение | ARS (public) |
|---|---|---|
| 1 | K-means, K = 10, на исходных 77 888 признаках | 0.038 |
| 2 | TruncatedSVD до 100 признаков + K-means, K = 10 | *в работе* |

## 1. Загрузка данных

Файл `.npz` — разреженная матрица scipy, а не таблица csv, поэтому читаем через `scipy.sparse.load_npz`, а не через pandas.

- `X.shape` — размер: строки × столбцы;
- `X.nnz` — число **ненулевых** клеток (number of non-zeros). Разреженная матрица хранит только их.

In [ ]:
from scipy.sparse import load_npz

X = load_npz("train.npz")

print(X.shape)   # сколько строк и столбцов
print(X.nnz)     # сколько ненулевых клеток

(14590, 77888)
2026769

В строке в среднем $2\,026\,769 / 14\,590 \approx 139$ ненулевых значений из 77 888 — заполнено около **0.18%** матрицы. Так выглядит **мешок слов**: строка — документ, столбец — слово, в документе ~139 разных слов из словаря.

## 2. Какие числа внутри

Если внутри целые числа (1, 2, 3) — это просто подсчёт слов (`CountVectorizer`). Если дробные — веса **TF-IDF**.

- `X[0].data` — ненулевые значения первой строки;
- `[:20]` — первые 20 из них.

In [ ]:
print(X[0].data[:20])

[0.14695191 0.1536871  0.16111354 0.1558444  0.14695191 0.1443309
 0.07967093 0.14099152 0.11611634 0.16111354 0.1558444  0.09661526
 0.14695191 0.04930589 0.12341241 0.14841796 0.14841796 0.11747369
 0.14695191 0.1365019 ]

Числа дробные — это **TF-IDF**:

$$
\text{TF-IDF}(\text{слово}, \text{документ}) = \underbrace{\text{TF}}_{\text{как часто слово в документе}} \times \underbrace{\log\frac{N}{\text{в скольких документах встречается слово}}}_{\text{IDF: насколько слово редкое}}
$$

Частые бесполезные слова («и», «в») получают маленький вес, редкие и характерные — большой.

## 3. Нормированы ли строки

TF-IDF обычно нормируют: делят строку на её длину, чтобы длина вектора стала 1. Тогда длинный и короткий тексты на одну тему окажутся рядом, а евклидово расстояние между векторами ведёт себя как **косинусное** — для текстов это то, что нужно.

Длина (норма) вектора:
$$
\|x\| = \sqrt{x_1^2 + x_2^2 + \dots + x_n^2}
$$

`norm(X, axis=1)` считает её для каждой строки (`axis=1` — «вдоль строки»).

In [ ]:
from scipy.sparse.linalg import norm

row_norms = norm(X, axis=1)
print(row_norms[:10])
print(row_norms.min(), row_norms.max())

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
0.9999999999999981 1.0000000000000009

Все длины равны 1 (отличия в 15-м знаке — погрешность вычислений). Строки **уже нормированы**, делать ничего не нужно.

**Итог разведки:** 14 590 текстов, каждый — нормированный TF-IDF-вектор длиной 77 888.

## 4. K-means

Нужно заранее задать **K** — на сколько групп делить.

1. Случайно поставить K центров.
2. Каждую точку отнести к **ближайшему** центру.
3. Сдвинуть каждый центр в **среднее** своих точек.
4. Повторять шаги 2–3, пока центры двигаются.

K-means минимизирует сумму квадратов расстояний от точек до центров их кластеров:
$$
\sum_{k=1}^{K} \sum_{x_i \in C_k} \|x_i - \mu_k\|^2 \to \min
$$
где $C_k$ — точки кластера $k$, $\mu_k$ — его центр. Центр — **среднее** точек, потому что среднее минимизирует сумму квадратов отклонений (как оптимальная константа для MSE).

K в задаче неизвестен. Для первого запуска берём K = 10.

- `random_state=42` — фиксируем случайную расстановку центров, чтобы результат повторялся;
- `n_init=5` — 5 запусков с разных стартов, берём лучший: K-means зависит от начальных центров;
- `fit_predict` — обучиться и сразу выдать номер кластера каждому объекту;
- `np.bincount(labels)` — размеры кластеров.

In [ ]:
from sklearn.cluster import KMeans
import numpy as np

kmeans = KMeans(n_clusters=10, random_state=42, n_init=5)
labels = kmeans.fit_predict(X)

print(labels[:20])           # номера кластеров первых 20 текстов
print(np.bincount(labels))   # сколько текстов в каждом кластере

[9 9 7 5 4 4 5 4 4 6 9 5 5 0 7 1 4 4 4 4]
[1481 1183  706  388 3966 2143 1168  952  411 2192]

Кластер 4 огромный — 3 966 текстов, **27%** всех. Похоже на «свалку»: туда K-means складывает тексты, которые не смог уверенно отнести никуда. Остальные кластеры — от 388 до 2 192.

Хорошее ли это разбиение, без правильных меток не понять. Сдаём как baseline.

## 5. Первый сабмит

Правильные `ID` берём из `sample_submission.csv`, чтобы не ошибиться с нумерацией, и сверяем длины: строк в примере должно быть столько же, сколько у нас меток. Порядок строк совпадает: i-я строка матрицы = i-й `ID`.

In [ ]:
import pandas as pd

sub = pd.read_csv("sample_submission.csv")
print(len(sub), "строк в примере,", len(labels), "меток у нас")

sub["TARGET"] = labels
sub.to_csv("kmeans_k10.csv", index=False)   # index=False — без лишней колонки с номерами строк
print(sub.head())

14590 строк в примере, 14590 меток у нас
   ID  TARGET
0   0       9
1   1       9
2   2       7
3   3       5
4   4       4

**ARS на public: 0.038** — почти как случайное разбиение.

**Почему так плохо — проклятие размерности.** С ростом числа признаков расстояния между всеми объектами становятся почти одинаковыми. У нас 77 888 признаков, а K-means работает на расстояниях: в таком пространстве все тексты «примерно одинаково далеко» друг от друга, и настоящие группы найти трудно.

## 6. Снижение размерности: TruncatedSVD (LSA) — *следующий шаг*

**Идея:** сжать 77 888 признаков до 100, сохранив главное. **TruncatedSVD** — аналог PCA для разреженных матриц; для текстов его называют **LSA** (латентный семантический анализ). Он объединяет слова, которые часто встречаются вместе, в «темы»: «гол», «матч», «тренер» → один признак «про футбол». У каждого текста вместо 77 888 слов будет 100 чисел — насколько он про каждую тему.

После сжатия векторы снова нормируем до длины 1 (`Normalizer`) — по той же причине, что в шаге 3.

**K оставляем 10:** меняем одну вещь за раз.

- `explained_variance_ratio_.sum()` — какую долю разброса исходных данных сохранили 100 новых признаков. Для текстов это обычно 10–20%: выбрасываем в основном шум и редкие слова.

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer

svd = TruncatedSVD(n_components=100, random_state=42)
X_svd = svd.fit_transform(X)                  # 77 888 признаков → 100
X_svd = Normalizer().fit_transform(X_svd)     # снова длина каждой строки = 1

print(X_svd.shape)
print("доля сохранённой информации:", svd.explained_variance_ratio_.sum())

kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_svd)
print(np.bincount(labels))

sub["TARGET"] = labels
sub.to_csv("svd100_kmeans_k10.csv", index=False)

## Что дальше

| Идея | Зачем |
|---|---|
| Подобрать K | настоящее число классов неизвестно; пробуем разные K и сравниваем по лидерборду и по внутренним метрикам (silhouette) |
| Подобрать число SVD-компонент | 50 / 100 / 200 — компромисс между шумом и потерей информации |
| Другие алгоритмы (агломеративная кластеризация, спектральная) | K-means предполагает «круглые» кластеры одного размера |